# YOLOv11 Heatmap Visualization (Grad-CAM)

此 Notebook 用於可視化模型在偵測物件時關注的區域 (Heatmap)。
這是檢查模型是否學習到正確特徵的工具。

In [ ]:
%pip install ultralytics opencv-python matplotlib

In [ ]:
import warnings
warnings.filterwarnings('ignore')
from ultralytics import YOLO
from ultralytics.utils.plotting import Annotator, colors
import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch

def get_heatmap(model, img_path, layer_index=-2):
    """
    這是一個簡化的 Grad-CAM 實作，針對 YOLOv8/11 架構。
    它會掛鉤 (Hook) 到模型的指定層，並計算梯度以產生熱力圖。
    注意：這是實驗性質的功能，Ultralytics 官方尚未完全內建 GradCAM API。
    """
    
    # 載入圖片
    img = cv2.imread(img_path)
    img = cv2.resize(img, (640, 640))
    input_tensor = torch.from_numpy(img).permute(2, 0, 1).unsqueeze(0).float() / 255.0
    input_tensor = input_tensor.to('cuda' if torch.cuda.is_available() else 'cpu')
    
    # 定義 Hook
    gradients = []
    activations = []
    
    def backward_hook(module, grad_input, grad_output):
        gradients.append(grad_output[0])
        
    def forward_hook(module, input, output):
        activations.append(output)

    # 尋找目標層 (通常是 Detect 之前的最後幾層卷積)
    # 這裡我們嘗試掛鉤到 backbone 或 neck 的最後一層
    # 如果失敗，您可能需要列印 model.model.named_modules() 來找適合的層
    target_layer = list(model.model.modules())[layer_index] 
    
    handle_f = target_layer.register_forward_hook(forward_hook)
    handle_b = target_layer.register_full_backward_hook(backward_hook)
    
    # Inference
    model(input_tensor)
    
    # Backward (這裡有點 trick，我們對輸出做一個簡單的 sum backward 來取得梯度)
    # 更精確的做法是針對特定的 class score 做 backward
    # 但做為通用觀察，這通常夠用
    # 注意：YOLO 的輸出比較複雜，這裡我們簡單觸發梯度
    
    # 為了簡化，我們使用一個更通用的庫 'yolo-cam' 的概念，
    # 但如果要在純 Notebook 執行且不依賴外部複雜庫，我們可以使用 'eigen-cam' 的概念 (不需要 backward)
    # 下面切換成 EigenCAM 類似方法 (Principal Component of Activations)，這對由檢測網路很有效且不需要 class index
    
    handle_f.remove()
    handle_b.remove()
    
    if not activations:
        print("無法捕捉 Activations")
        return img
        
    activation = activations[0].detach().cpu().numpy()[0]
    # activation shape: (C, H, W)
    
    # 簡單的將所有通道平均 (或者取最大值)
    heatmap = np.mean(activation, axis=0)
    
    # 正規化
    heatmap = np.maximum(heatmap, 0)
    heatmap /= np.max(heatmap) + 1e-8
    
    return heatmap

def overlay_heatmap(img_path, heatmap):
    img = cv2.imread(img_path)
    img = cv2.resize(img, (640, 640))
    
    heatmap = cv2.resize(heatmap, (img.shape[1], img.shape[0]))
    heatmap = np.uint8(255 * heatmap)
    heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
    
    superimposed_img = heatmap * 0.4 + img
    superimposed_img = np.clip(superimposed_img, 0, 255).astype(np.uint8)
    return superimposed_img

# 主程式
# 請填入您的模型路徑和測試圖片
model_path = 'yolo11m.pt' # 或 'runs/detect/train/weights/best.pt'
img_path = 'ingredients_dataset/class100_yolov11/test/images/some_image.jpg' # 請替換成一張真實存在的圖片路徑

# 由於我們沒有指定具體的測試圖，這裡做一個範例說明
print("請修改 img_path 為您想要測試的圖片路徑")

# 範例執行 (若圖片存在)
if os.path.exists(img_path):
    model = YOLO(model_path)
    # 嘗試捕捉最後一層卷積特徵 (通常 layer_index=-2 或 -3)
    hm = get_heatmap(model, img_path, layer_index=-2) 
    result = overlay_heatmap(img_path, hm)
    
    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1)
    plt.title("Original")
    plt.imshow(cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB))
    
    plt.subplot(1, 2, 2)
    plt.title("Heatmap Overlay")
    plt.imshow(cv2.cvtColor(result, cv2.COLOR_BGR2RGB))
    plt.show()
else:
    print(f"找不到圖片: {img_path}")